# Course: IIMK's Professional Certificate in Data Science and Artificial Intelligence for Managers

**Student Name:** Lalit Nayyar  
**Email ID:** lalitnayyar@gmail.com  
**Assignment Name:** Week 3: Required Assignment 3.1

---

In [ ]:
# Install required packages
!pip install pandas numpy matplotlib seaborn scikit-learn kaggle folium --upgrade

In [ ]:
# Import required libraries
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.model_selection import train_test_split, cross_val_score
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LinearRegression, Ridge, Lasso
from sklearn.metrics import mean_squared_error, r2_score
from sklearn.cluster import KMeans
import warnings
warnings.filterwarnings('ignore')

# Configure plotting settings
plt.rcParams['figure.figsize'] = [10, 6]
plt.rcParams['figure.dpi'] = 100
plt.rcParams['font.size'] = 12
plt.rcParams['axes.grid'] = True

# Set seaborn style safely
try:
    sns.set_theme(style="whitegrid")
except Exception as e:
    print(f"Note: Could not set seaborn theme. Using default matplotlib style. Error: {e}")

### Download and Load the Dataset

In [ ]:
# Download the dataset using Kaggle API
import os
from kaggle.api.kaggle_api_extended import KaggleApi

def download_dataset():
    try:
        api = KaggleApi()
        api.authenticate()
        
        print("Downloading California Housing Prices dataset...")
        api.dataset_download_files('camnugent/california-housing-prices', 
                                  path='.', 
                                  unzip=True)
        print("Dataset downloaded successfully!")
        
    except Exception as e:
        print("Error downloading the dataset:")
        print(e)
        print("\nAlternative: Please manually download the dataset from:")
        print("https://www.kaggle.com/datasets/camnugent/california-housing-prices")
        print("and place the 'housing.csv' file in the current directory.")

# Download the dataset if it doesn't exist
if not os.path.exists('housing.csv'):
    download_dataset()

In [ ]:
# Load the dataset
df = pd.read_csv('housing.csv')

# Display basic information
print("Dataset Info:")
print(df.info())
print("\nFirst few rows:")
display(df.head())

## Real Estate Agent Analysis

In [ ]:
# 1. Region-specific Price Analysis

# Create geographical regions
df['longitude_region'] = pd.qcut(df['longitude'], q=5, labels=['West Coast', 'West Central', 'Central', 'East Central', 'East'])
df['latitude_region'] = pd.qcut(df['latitude'], q=3, labels=['South', 'Central', 'North'])

# Calculate regional statistics
regional_stats = df.groupby(['longitude_region', 'latitude_region']).agg({
    'median_house_value': ['mean', 'std'],
    'median_income': 'mean',
    'housing_median_age': 'mean'
}).round(2)

# Visualize regional price variations
fig, ax = plt.subplots(figsize=(15, 10))
for region in df['longitude_region'].unique():
    region_data = df[df['longitude_region'] == region]
    scatter = ax.scatter(region_data['longitude'], 
                        region_data['latitude'], 
                        c=region_data['median_house_value'],
                        alpha=0.6,
                        label=region)

plt.colorbar(scatter, label='Median House Value')
ax.set_title('Regional Price Distribution')
ax.legend()
plt.show()

# Display regional statistics
print("\nRegional Market Statistics:")
display(regional_stats)

In [ ]:
# 2. Local Market Factor Analysis

# Calculate local market indicators
df['price_per_room'] = df['median_house_value'] / df['total_rooms']
df['occupancy_rate'] = df['households'] / df['total_rooms']
df['income_to_house_ratio'] = df['median_income'] / (df['median_house_value'] / 100000)

# Create subplots
fig, axes = plt.subplots(2, 2, figsize=(15, 12))
fig.suptitle('Local Market Analysis', fontsize=14)

# Price vs Income by Region
for region in df['longitude_region'].unique():
    region_data = df[df['longitude_region'] == region]
    axes[0,0].scatter(region_data['median_income'], 
                      region_data['median_house_value'],
                      alpha=0.6,
                      label=region)
axes[0,0].set_title('Price vs Income by Region')
axes[0,0].set_xlabel('Median Income')
axes[0,0].set_ylabel('Median House Value')
axes[0,0].legend()

# Occupancy Rate Distribution
df.boxplot(column='occupancy_rate', by='longitude_region', ax=axes[0,1])
axes[0,1].set_title('Occupancy Rate by Region')
axes[0,1].tick_params(axis='x', rotation=45)

# Price per Room Distribution
df.boxplot(column='price_per_room', by='longitude_region', ax=axes[1,0])
axes[1,0].set_title('Price per Room by Region')
axes[1,0].tick_params(axis='x', rotation=45)

# Income to House Price Ratio
df.boxplot(column='income_to_house_ratio', by='longitude_region', ax=axes[1,1])
axes[1,1].set_title('Income to House Price Ratio by Region')
axes[1,1].tick_params(axis='x', rotation=45)

plt.tight_layout()
plt.show()

[Rest of the notebook sections remain the same...]